In [ ]:
# Cell 1: imports
import ast
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from captum.attr import IntegratedGradients

In [ ]:
# Cell 2: load data safely
data_df = pd.read_csv("masking_data/blb_machine_clusters.tsv", sep="\t")
data_df = data_df.sort_values(by="slave_1760_1900", ascending=False).reset_index(drop=True)
data_df.head()

In [ ]:
# Cell 3: helpers
def pick_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

def parse_pred_column(cell, top_n=5):
    # expected format: "[('word1', score1), ('word2', score2), ...]"
    items = ast.literal_eval(cell)
    return [w for w, _ in items[:top_n]]

def build_texts_targets(df, start, end, pred_col, top_n=5):
    subset = df.iloc[start:end].copy()
    texts = subset["maskedSentence"].tolist()
    targets = [parse_pred_column(x, top_n=top_n) for x in subset[pred_col].tolist()]
    return texts, targets

In [ ]:
# ...existing code...
class MaskedLMExplainer:
    def __init__(self, model_name="bert-base-uncased", device=None):
        self.device = device or pick_device()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForMaskedLM.from_pretrained(model_name).to(self.device)
        self.model.eval()
        self.ig = IntegratedGradients(self.forward_func)

    # Supports K mask positions / K target tokens per example
    def forward_func(self, inputs_embeds, attention_mask, mask_indices, target_indices, valid_positions):
        logits = self.model(inputs_embeds=inputs_embeds, attention_mask=attention_mask).logits  # (B, L, V)
        bsz = logits.size(0)

        # (B, K, V): logits at each masked position
        mask_logits = logits[
            torch.arange(bsz, device=logits.device).unsqueeze(1),
            mask_indices
        ]
        log_probs = torch.log_softmax(mask_logits, dim=-1)  # (B, K, V)

        # (B, K): log p(target_token_k | context)
        target_lp = log_probs.gather(-1, target_indices.unsqueeze(-1)).squeeze(-1)

        # Sum only valid positions
        return (target_lp * valid_positions).sum(dim=1)  # (B,)

    def _target_to_token_ids(self, text_target):
        ids = self.tokenizer(text_target, add_special_tokens=False)["input_ids"]
        return ids

    def _expand_single_mask(self, text, n_masks):
        mask = self.tokenizer.mask_token
        if text.count(mask) != 1:
            raise ValueError("Input sentence must contain exactly one [MASK] before expansion.")
        return text.replace(mask, " ".join([mask] * n_masks), 1)
    
    def _aggregate_tokens_to_words(self, token_rows, agg="mean"):
        """
        token_rows: list[(token, score)]
        Returns: list[(word, aggregated_score)]
        """
        if agg not in {"mean", "max"}:
            raise ValueError("agg must be 'mean' or 'max'")

        def reduce_scores(scores):
            return float(sum(scores) / len(scores)) if agg == "mean" else float(max(scores))

        word_rows = []
        current_word = None
        current_scores = []

        for tok, score in token_rows:
            # BERT WordPiece continuation
            if tok.startswith("##"):
                piece = tok[2:]
                if current_word is None:
                    current_word = piece
                    current_scores = [score]
                else:
                    current_word += piece
                    current_scores.append(score)
            else:
                if current_word is not None:
                    word_rows.append((current_word, reduce_scores(current_scores)))
                current_word = tok
                current_scores = [score]

        if current_word is not None:
            word_rows.append((current_word, reduce_scores(current_scores)))

        return word_rows

    def explain(
        self,
        texts,
        target_words_list,
        normalize=True,
        drop_special=True,
        return_word_scores=True,
        word_agg="mean",  # "mean" or "max"
    ):
        if len(texts) != len(target_words_list):
            raise ValueError("texts and target_words_list must have same length")

        all_results = []
        for text, targets in zip(texts, target_words_list):
            sent_out = {}

            for target in targets:
                target_ids = self._target_to_token_ids(target)
                if len(target_ids) == 0:
                    sent_out[target] = {"skipped": True, "reason": "empty tokenization"}
                    continue

                text_k = self._expand_single_mask(text, len(target_ids))
                enc = self.tokenizer([text_k], return_tensors="pt", padding=True, truncation=True)
                input_ids = enc["input_ids"].to(self.device)
                attention_mask = enc["attention_mask"].to(self.device)

                mask_pos = (input_ids[0] == self.tokenizer.mask_token_id).nonzero(as_tuple=False).flatten()
                if mask_pos.numel() != len(target_ids):
                    sent_out[target] = {
                        "skipped": True,
                        "reason": f"mask count ({mask_pos.numel()}) != target token count ({len(target_ids)})"
                    }
                    continue

                emb = self.model.get_input_embeddings()(input_ids)
                baseline_ids = torch.full_like(input_ids, self.tokenizer.pad_token_id)
                baseline_ids[0, mask_pos] = self.tokenizer.mask_token_id
                base = self.model.get_input_embeddings()(baseline_ids)

                mpos = mask_pos.unsqueeze(0)
                tid = torch.tensor(target_ids, device=self.device).unsqueeze(0)
                valid = torch.ones_like(tid, dtype=torch.float32, device=self.device)

                attrs = self.ig.attribute(
                    inputs=emb,
                    baselines=base,
                    additional_forward_args=(attention_mask, mpos, tid, valid),
                ).sum(dim=-1).squeeze(0)

                if normalize:
                    attrs = attrs / attrs.norm().clamp_min(1e-12)

                attrs[mask_pos] = 0.0

                tokens = self.tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
                token_rows = []
                for tok, val, tid_ in zip(tokens, attrs.tolist(), input_ids[0].tolist()):
                    if drop_special and tid_ in {
                        self.tokenizer.cls_token_id,
                        self.tokenizer.sep_token_id,
                        self.tokenizer.pad_token_id,
                    }:
                        continue
                    token_rows.append((tok, float(val)))

                result_obj = {
                    "skipped": False,
                    "target_token_ids": target_ids,
                    "token_attributions": token_rows,
                }

                if return_word_scores:
                    result_obj["word_attributions"] = self._aggregate_tokens_to_words(
                        token_rows, agg=word_agg
                    )

                sent_out[target] = result_obj

            all_results.append(sent_out)

        return all_results


In [ ]:
# Cell 5: run explainer
iloc_range = (0, 20)
texts, targets = build_texts_targets(
    data_df, start=iloc_range[0], end=iloc_range[1],
    pred_col="pred_bert_1760_1900", top_n=5
)

explainer = MaskedLMExplainer(model_name="Livingwithmachines/bert_1760_1900", device=pick_device())
results = explainer.explain(texts, targets, word_agg="max")


In [ ]:
# Cell 6: fixed model comparison
def compare_explainers(explainer_1, explainer_2, texts, targets, level="word", word_agg="mean"):
    r1 = explainer_1.explain(texts, targets, return_word_scores=(level == "word"), word_agg=word_agg)
    r2 = explainer_2.explain(texts, targets, return_word_scores=(level == "word"), word_agg=word_agg)

    key = "word_attributions" if level == "word" else "token_attributions"

    comparisons = []
    for i in range(len(texts)):
        out = {}
        for target in targets[i]:
            a = r1[i][target]
            b = r2[i][target]
            if a.get("skipped") or b.get("skipped"):
                out[target] = {"skipped": True, "model1": a, "model2": b}
                continue

            rows1 = a[key]
            rows2 = b[key]

            t1 = [x[0] for x in rows1]
            t2 = [x[0] for x in rows2]
            if t1 != t2:
                raise ValueError(f"{level.capitalize()} mismatch at sentence {i}, target '{target}'")

            s1 = [x[1] for x in rows1]
            s2 = [x[1] for x in rows2]
            out[target] = list(zip(t1, s1, s2, [x - y for x, y in zip(s1, s2)]))
        comparisons.append(out)
    return comparisons

explainer_new = MaskedLMExplainer("Livingwithmachines/bert_1760_1900", device=pick_device())
explainer_old = MaskedLMExplainer("Livingwithmachines/bert_1760_1850", device=pick_device())

# Correct shape: one target-list per sentence
single_target = [["slave"] for _ in texts]

comparison = compare_explainers(explainer_new, explainer_old, texts, single_target, level="word", word_agg="max")

comparison = compare_explainers(explainer_new, explainer_old, texts, single_target)

In [ ]:
# Cell 7: Summarize top predictors

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def summarize_top_predictors(results, target, top_n=10):
    """
    Aggregate attributions across all sentences for a single target word.
    
    Args:
        results: output from explainer.explain()
        target: target word to summarize
        top_n: number of top predictors to return
    
    Returns:
        list of (word, mean_attribution, std_attribution, count)
    """
    word_scores = {}
    
    for sent_result in results:
        if target not in sent_result or sent_result[target].get("skipped"):
            continue
        
        word_attrs = sent_result[target]["word_attributions"]
        for word, score in word_attrs:
            if word not in word_scores:
                word_scores[word] = []
            word_scores[word].append(score)
    
    # Compute statistics
    stats = []
    for word, scores in word_scores.items():
        mean_score = float(np.mean(scores))
        std_score = float(np.std(scores))
        count = len(scores)
        stats.append((word, mean_score, std_score, count))
    
    # Sort by mean attribution (descending)
    stats.sort(key=lambda x: abs(x[1]), reverse=True)
    
    return stats[:top_n]

# Example usage
top_predictors = summarize_top_predictors(results, target="slave", top_n=15)
print(f"Top predictors for 'slave':\n")
print(f"{'Word':>15} {'Mean':>10} {'Std':>10} {'Count':>8}")
print("-" * 45)
for word, mean, std, count in top_predictors:
    print(f"{word:>15} {mean:>10.4f} {std:>10.4f} {count:>8}")

In [ ]:
# Cell 8: Visualization methods

def plot_top_predictors_bar(results, target, top_n=15, metric="mean"):
    """
    Bar chart of top predictors (by absolute attribution).
    
    Args:
        results: output from explainer.explain()
        target: target word
        top_n: number of top words to show
        metric: "mean" or "max"
    """
    stats = summarize_top_predictors(results, target, top_n=top_n)
    
    words = [s[0] for s in stats]
    means = [s[1] for s in stats]
    stds = [s[2] for s in stats]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = ["green" if m > 0 else "red" for m in means]
    ax.barh(words, means, xerr=stds, color=colors, alpha=0.7, capsize=5)
    
    ax.set_xlabel("Mean Attribution Score", fontsize=12)
    ax.set_title(f"Top {top_n} Predictors for '{target}'", fontsize=14, fontweight="bold")
    ax.axvline(x=0, color="black", linestyle="--", linewidth=0.8)
    ax.grid(axis="x", alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_attribution_heatmap_sentences(results, target, sent_indices=None, figsize=(14, 8)):
    """
    Heatmap showing attributions for each sentence (rows) and words (columns).
    
    Args:
        results: output from explainer.explain()
        target: target word
        sent_indices: list of sentence indices to include (None = all)
        figsize: figure size
    """
    word_set = set()
    sent_data = []
    
    for i, sent_result in enumerate(results):
        if target not in sent_result or sent_result[target].get("skipped"):
            continue
        
        if sent_indices is not None and i not in sent_indices:
            continue
        
        word_attrs = sent_result[target]["word_attributions"]
        word_dict = {word: score for word, score in word_attrs}
        word_set.update(word_dict.keys())
        sent_data.append((i, word_dict))
    
    # Filter to most important words
    word_importance = {}
    for _, word_dict in sent_data:
        for word, score in word_dict.items():
            if word not in word_importance:
                word_importance[word] = 0
            word_importance[word] += abs(score)
    
    top_words = sorted(word_importance.items(), key=lambda x: x[1], reverse=True)[:20]
    top_words = [w[0] for w in top_words]
    
    # Build matrix
    matrix = []
    sent_labels = []
    for sent_idx, word_dict in sent_data:
        row = [word_dict.get(w, 0) for w in top_words]
        matrix.append(row)
        sent_labels.append(f"S{sent_idx}")
    
    matrix = np.array(matrix)
    
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(matrix, cmap="RdBu_r", aspect="auto", vmin=-np.max(np.abs(matrix)), vmax=np.max(np.abs(matrix)))
    
    ax.set_xticks(range(len(top_words)))
    ax.set_xticklabels(top_words, rotation=45, ha="right")
    ax.set_yticks(range(len(sent_labels)))
    ax.set_yticklabels(sent_labels)
    
    ax.set_title(f"Attribution Heatmap for '{target}' Across Sentences", fontsize=14, fontweight="bold")
    ax.set_xlabel("Context Words", fontsize=12)
    ax.set_ylabel("Sentences", fontsize=12)
    
    plt.colorbar(im, ax=ax, label="Attribution Score")
    plt.tight_layout()
    plt.show()

def plot_distribution_violin(results, target, top_n=10):
    """
    Violin plot showing distribution of attributions for top words.
    
    Args:
        results: output from explainer.explain()
        target: target word
        top_n: number of top words to show
    """
    stats = summarize_top_predictors(results, target, top_n=top_n)
    top_words = [s[0] for s in stats]
    
    # Collect all scores per word
    word_scores = {w: [] for w in top_words}
    
    for sent_result in results:
        if target not in sent_result or sent_result[target].get("skipped"):
            continue
        
        word_attrs = sent_result[target]["word_attributions"]
        word_dict = {word: score for word, score in word_attrs}
        
        for word in top_words:
            if word in word_dict:
                word_scores[word].append(word_dict[word])
    
    # Prepare data for violin plot
    plot_data = []
    labels = []
    for word in top_words:
        plot_data.extend(word_scores[word])
        labels.extend([word] * len(word_scores[word]))
    
    fig, ax = plt.subplots(figsize=(12, 6))
    parts = ax.violinplot(
        [word_scores[w] for w in top_words],
        positions=range(len(top_words)),
        showmeans=True,
        showmedians=True
    )
    
    ax.set_xticks(range(len(top_words)))
    ax.set_xticklabels(top_words, rotation=45, ha="right")
    ax.set_ylabel("Attribution Score", fontsize=12)
    ax.set_title(f"Distribution of Attributions for Top Predictors of '{target}'", fontsize=14, fontweight="bold")
    ax.axhline(y=0, color="black", linestyle="--", linewidth=0.8)
    ax.grid(axis="y", alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_word_cloud_style(results, target, top_n=25):
    """
    Word-cloud style visualization using word size for importance.
    
    Args:
        results: output from explainer.explain()
        target: target word
        top_n: number of words to show
    """
    from textwrap import wrap
    
    stats = summarize_top_predictors(results, target, top_n=top_n)
    
    words = [s[0] for s in stats]
    scores = np.array([abs(s[1]) for s in stats])
    
    # Normalize scores for sizing
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    sizes = 20 + scores_norm * 200
    
    colors_val = [s[1] for s in stats]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    scatter = ax.scatter(
        range(len(words)), [1]*len(words),
        s=sizes, c=colors_val, cmap="RdBu_r", alpha=0.6,
        edgecolors="black", linewidth=1
    )
    
    for i, word in enumerate(words):
        ax.text(i, 1, word, ha="center", va="center", fontsize=max(8, int(sizes[i]/15)), fontweight="bold")
    
    ax.set_xlim(-1, len(words))
    ax.set_ylim(0.5, 1.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)
    
    ax.set_title(f"Top {top_n} Predictors for '{target}' (by importance)", fontsize=14, fontweight="bold")
    
    plt.colorbar(scatter, ax=ax, label="Attribution Score")
    plt.tight_layout()
    plt.show()

In [ ]:

# Cell 9: Generate visualizations

# 1. Bar chart
plot_top_predictors_bar(results, target="slave", top_n=15)

# 2. Heatmap across sentences
plot_attribution_heatmap_sentences(results, target="slave")

# 3. Violin plot (distribution)
plot_distribution_violin(results, target="slave", top_n=12)

# 4. Word-cloud style
plot_word_cloud_style(results, target="slave", top_n=20)

# 5. Summary table to CSV
stats = summarize_top_predictors(results, target="slave", top_n=20)
summary_df = pd.DataFrame(stats, columns=["Word", "Mean_Attribution", "Std_Attribution", "Count"])
summary_df.to_csv("top_predictors_slave.csv", index=False)
summary_df

In [ ]:
# Cell 10: Analyze model comparison

def analyze_comparison(comparison, target, top_n=15):
    """
    Summarize differences between two models across all sentences.
    
    Args:
        comparison: output from compare_explainers()
        target: target word
        top_n: number of top differences to return
    
    Returns:
        list of (word, model1_score, model2_score, difference, abs_difference)
    """
    word_diffs = {}
    
    for sent_comp in comparison:
        if target not in sent_comp or sent_comp[target].get("skipped"):
            continue
        
        word_tuples = sent_comp[target]
        for word, s1, s2, diff in word_tuples:
            if word not in word_diffs:
                word_diffs[word] = []
            word_diffs[word].append((s1, s2, diff))
    
    # Aggregate statistics
    stats = []
    for word, deltas in word_diffs.items():
        s1_vals = [x[0] for x in deltas]
        s2_vals = [x[1] for x in deltas]
        diff_vals = [x[2] for x in deltas]
        
        mean_s1 = float(np.mean(s1_vals))
        mean_s2 = float(np.mean(s2_vals))
        mean_diff = float(np.mean(diff_vals))
        std_diff = float(np.std(diff_vals))
        
        stats.append((word, mean_s1, mean_s2, mean_diff, std_diff))
    
    # Sort by absolute difference
    stats.sort(key=lambda x: abs(x[3]), reverse=True)
    
    return stats[:top_n]

# Example usage
comparison_stats = analyze_comparison(comparison, target="slave", top_n=15)
print(f"Top diverging words between models for 'slave':\n")
print(f"{'Word':>15} {'Model1':>12} {'Model2':>12} {'Diff':>12} {'Std':>10}")
print("-" * 62)
for word, m1, m2, diff, std in comparison_stats:
    print(f"{word:>15} {m1:>12.4f} {m2:>12.4f} {diff:>12.4f} {std:>10.4f}")

In [ ]:
# Cell 11: Comparison visualization methods

def plot_model_comparison_bar(comparison, target, top_n=15):
    """
    Side-by-side bar chart comparing model attributions.
    """
    stats = analyze_comparison(comparison, target, top_n=top_n)
    
    words = [s[0] for s in stats]
    m1_scores = [s[1] for s in stats]
    m2_scores = [s[2] for s in stats]
    
    x = np.arange(len(words))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    bars1 = ax.barh(x - width/2, m1_scores, width, label="Model 1 (1760-1900)", alpha=0.8)
    bars2 = ax.barh(x + width/2, m2_scores, width, label="Model 2 (1760-1850)", alpha=0.8)
    
    ax.set_yticks(x)
    ax.set_yticklabels(words)
    ax.set_xlabel("Attribution Score", fontsize=12)
    ax.set_title(f"Model Comparison: Top {top_n} Predictors for '{target}'", fontsize=14, fontweight="bold")
    ax.axvline(x=0, color="black", linestyle="--", linewidth=0.8)
    ax.legend(fontsize=11)
    ax.grid(axis="x", alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_difference_heatmap(comparison, target, figsize=(14, 8)):
    """
    Heatmap of differences (Model2 - Model1) across sentences.
    Positive = Model2 stronger, Negative = Model1 stronger
    """
    sent_data = []
    word_set = set()
    
    for i, sent_comp in enumerate(comparison):
        if target not in sent_comp or sent_comp[target].get("skipped"):
            continue
        
        word_tuples = sent_comp[target]
        word_dict = {word: diff for word, s1, s2, diff in word_tuples}
        word_set.update(word_dict.keys())
        sent_data.append((i, word_dict))
    
    # Get top words by total absolute difference
    word_importance = {}
    for _, word_dict in sent_data:
        for word, diff in word_dict.items():
            if word not in word_importance:
                word_importance[word] = 0
            word_importance[word] += abs(diff)
    
    top_words = sorted(word_importance.items(), key=lambda x: x[1], reverse=True)[:20]
    top_words = [w[0] for w in top_words]
    
    # Build matrix
    matrix = []
    sent_labels = []
    for sent_idx, word_dict in sent_data:
        row = [word_dict.get(w, 0) for w in top_words]
        matrix.append(row)
        sent_labels.append(f"S{sent_idx}")
    
    matrix = np.array(matrix)
    vmax = np.max(np.abs(matrix))
    
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(matrix, cmap="RdBu_r", aspect="auto", vmin=-vmax, vmax=vmax)
    
    ax.set_xticks(range(len(top_words)))
    ax.set_xticklabels(top_words, rotation=45, ha="right")
    ax.set_yticks(range(len(sent_labels)))
    ax.set_yticklabels(sent_labels)
    
    ax.set_title(f"Attribution Difference: Model2 - Model1 for '{target}'", fontsize=14, fontweight="bold")
    ax.set_xlabel("Context Words", fontsize=12)
    ax.set_ylabel("Sentences", fontsize=12)
    
    cbar = plt.colorbar(im, ax=ax, label="Difference (Model2 - Model1)")
    plt.tight_layout()
    plt.show()

def plot_scatter_model_comparison(comparison, target, top_n=25):
    """
    Scatter plot: Model1 scores vs Model2 scores.
    Points above diagonal = Model2 favors this word more.
    Points below diagonal = Model1 favors this word more.
    """
    sent_data = []
    
    for sent_comp in comparison:
        if target not in sent_comp or sent_comp[target].get("skipped"):
            continue
        
        word_tuples = sent_comp[target]
        for word, s1, s2, diff in word_tuples:
            sent_data.append((word, s1, s2))
    
    # Aggregate by word (mean scores)
    word_agg = {}
    for word, s1, s2 in sent_data:
        if word not in word_agg:
            word_agg[word] = {"s1": [], "s2": []}
        word_agg[word]["s1"].append(s1)
        word_agg[word]["s2"].append(s2)
    
    word_means = []
    for word, vals in word_agg.items():
        m1 = np.mean(vals["s1"])
        m2 = np.mean(vals["s2"])
        diff = abs(m2 - m1)
        word_means.append((word, m1, m2, diff))
    
    # Sort by difference and take top N
    word_means.sort(key=lambda x: x[3], reverse=True)
    word_means = word_means[:top_n]
    
    words = [w[0] for w in word_means]
    m1_vals = [w[1] for w in word_means]
    m2_vals = [w[2] for w in word_means]
    diffs = [w[3] for w in word_means]
    
    fig, ax = plt.subplots(figsize=(10, 10))
    
    scatter = ax.scatter(m1_vals, m2_vals, s=200, c=diffs, cmap="YlOrRd", alpha=0.6, edgecolors="black", linewidth=1)
    
    for i, word in enumerate(words):
        ax.annotate(word, (m1_vals[i], m2_vals[i]), fontsize=9, ha="center", va="center")
    
    # Diagonal line (where scores are equal)
    lim_min = min(min(m1_vals), min(m2_vals)) * 0.9
    lim_max = max(max(m1_vals), max(m2_vals)) * 1.1
    ax.plot([lim_min, lim_max], [lim_min, lim_max], "k--", alpha=0.3, linewidth=2, label="Equal scores")
    
    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)
    ax.set_xlabel("Model 1 Attribution Score (1760-1900)", fontsize=12)
    ax.set_ylabel("Model 2 Attribution Score (1760-1850)", fontsize=12)
    ax.set_title(f"Model Attribution Comparison for '{target}'", fontsize=14, fontweight="bold")
    ax.grid(alpha=0.3)
    
    cbar = plt.colorbar(scatter, ax=ax, label="Absolute Difference")
    ax.legend()
    
    plt.tight_layout()
    plt.show()

def plot_divergence_ranking(comparison, target, top_n=15):
    """
    Show which words changed most in relative ranking between models.
    """
    # Collect word scores per sentence
    sent_rankings = []
    
    for sent_comp in comparison:
        if target not in sent_comp or sent_comp[target].get("skipped"):
            continue
        
        word_tuples = sent_comp[target]
        m1_dict = {word: s1 for word, s1, s2, diff in word_tuples}
        m2_dict = {word: s2 for word, s1, s2, diff in word_tuples}
        
        # Rank by absolute score
        m1_ranked = sorted(m1_dict.items(), key=lambda x: abs(x[1]), reverse=True)
        m2_ranked = sorted(m2_dict.items(), key=lambda x: abs(x[1]), reverse=True)
        
        m1_ranks = {word: i for i, (word, _) in enumerate(m1_ranked)}
        m2_ranks = {word: i for i, (word, _) in enumerate(m2_ranked)}
        
        sent_rankings.append((m1_ranks, m2_ranks))
    
    # Compute average rank change
    all_words = set()
    for m1_ranks, m2_ranks in sent_rankings:
        all_words.update(m1_ranks.keys())
        all_words.update(m2_ranks.keys())
    
    rank_changes = {}
    for word in all_words:
        m1_r = [m1_ranks.get(word, len(m1_ranks)) for m1_ranks, _ in sent_rankings]
        m2_r = [m2_ranks.get(word, len(m2_ranks)) for _, m2_ranks in sent_rankings]
        
        avg_m1_rank = np.mean(m1_r)
        avg_m2_rank = np.mean(m2_r)
        rank_change = avg_m2_rank - avg_m1_rank  # negative = rose in ranking
        
        rank_changes[word] = (avg_m1_rank, avg_m2_rank, rank_change)
    
    # Sort by absolute rank change
    rank_changes = sorted(rank_changes.items(), key=lambda x: abs(x[1][2]), reverse=True)[:top_n]
    
    words = [w[0] for w in rank_changes]
    m1_ranks = [w[1][0] for w in rank_changes]
    m2_ranks = [w[1][1] for w in rank_changes]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    x = np.arange(len(words))
    width = 0.35
    
    ax.barh(x - width/2, m1_ranks, width, label="Model 1 Avg Rank", alpha=0.8)
    ax.barh(x + width/2, m2_ranks, width, label="Model 2 Avg Rank", alpha=0.8)
    
    ax.set_yticks(x)
    ax.set_yticklabels(words)
    ax.set_xlabel("Average Rank (lower = more important)", fontsize=12)
    ax.set_title(f"Ranking Changes: '{target}' Predictors Between Models", fontsize=14, fontweight="bold")
    ax.invert_xaxis()  # Lower rank (more important) on right
    ax.legend(fontsize=11)
    ax.grid(axis="x", alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def export_comparison_csv(comparison, target, output_file="model_comparison.csv"):
    """
    Export comparison results to CSV for external analysis.
    """
    rows = []
    
    for sent_idx, sent_comp in enumerate(comparison):
        if target not in sent_comp or sent_comp[target].get("skipped"):
            continue
        
        word_tuples = sent_comp[target]
        for word, s1, s2, diff in word_tuples:
            rows.append({
                "sentence_idx": sent_idx,
                "word": word,
                "model1_score": float(s1),
                "model2_score": float(s2),
                "difference": float(diff),
            })
    
    comp_df = pd.DataFrame(rows)
    comp_df.to_csv(output_file, index=False)
    print(f"Comparison exported to {output_file}")
    return comp_df

In [ ]:
# Cell 12: Generate comparison visualizations

# 1. Side-by-side bar chart
plot_model_comparison_bar(comparison, target="slave", top_n=15)

# 2. Difference heatmap
plot_difference_heatmap(comparison, target="slave")

# 3. Scatter plot (correlation)
plot_scatter_model_comparison(comparison, target="slave", top_n=25)

# 4. Ranking changes
plot_divergence_ranking(comparison, target="slave", top_n=15)

# 5. Export to CSV
comp_df = export_comparison_csv(comparison, target="slave", output_file="model_comparison_slave.csv")
comp_df.describe()